[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/volume2_chapter_05/notebook_5A_alert_design.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---

# Notebook 5A: Clinical Alert Design and Optimization

**Volume 2, Chapter 5: Deployment & Integration**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Understand the sensitivity-specificity tradeoff in clinical alerts
2. Calculate positive predictive value (PPV) at different disease prevalence levels
3. Simulate alert fatigue and its impact on clinician response
4. Design alert thresholds appropriate for different clinical contexts
5. Evaluate alert burden and optimize for clinical utility

## Clinical Context

Clinical decision support (CDS) alerts are a double-edged sword. Well-designed alerts can catch deteriorating patients, prevent medication errors, and improve outcomes. Poorly designed alerts create noise, desensitize clinicians, and paradoxically increase risk by burying important signals in a flood of false alarms.

**Key insight from Chapter 5:** The fundamental challenge of clinical alerts is that even highly accurate tests produce mostly false positives when applied to low-prevalence conditions. This is not a flaw in the test---it is a mathematical certainty that must inform alert design.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Define consistent color scheme
COLORS = {
    'reference': '#2E86AB',
    'current': '#E94F37',
    'alert': '#F39C12',
    'safe': '#27AE60'
}

print("Libraries imported successfully")

---

## 1. The PPV Problem: Why Most Alerts Are False

### Bayes' Theorem and Alert Design

When an alert fires, what is the probability that the patient truly has the condition? This is the **Positive Predictive Value (PPV)**, and it depends critically on three factors:

- **Sensitivity** (True Positive Rate): P(Alert | Disease)
- **Specificity** (True Negative Rate): P(No Alert | No Disease)
- **Prevalence**: P(Disease) in the population being screened

The relationship is given by Bayes' theorem:

$$PPV = \frac{Sensitivity \times Prevalence}{Sensitivity \times Prevalence + (1 - Specificity) \times (1 - Prevalence)}$$

**The critical insight:** Even with 95% sensitivity and 95% specificity, if prevalence is 1%, your PPV is only about 16%. This means **84% of alerts are false positives**.

In [ ]:
def calculate_ppv(sensitivity, specificity, prevalence):
    """
    Calculate Positive Predictive Value using Bayes' theorem.
    
    Parameters:
    -----------
    sensitivity : float
        True positive rate (0-1)
    specificity : float
        True negative rate (0-1)
    prevalence : float
        Disease prevalence in population (0-1)
    
    Returns:
    --------
    float : Positive Predictive Value (0-1)
    """
    numerator = sensitivity * prevalence
    denominator = numerator + (1 - specificity) * (1 - prevalence)
    
    if denominator == 0:
        return 0.0
    
    return numerator / denominator


def calculate_npv(sensitivity, specificity, prevalence):
    """
    Calculate Negative Predictive Value.
    
    Parameters:
    -----------
    sensitivity : float
        True positive rate (0-1)
    specificity : float
        True negative rate (0-1)
    prevalence : float
        Disease prevalence in population (0-1)
    
    Returns:
    --------
    float : Negative Predictive Value (0-1)
    """
    numerator = specificity * (1 - prevalence)
    denominator = numerator + (1 - sensitivity) * prevalence
    
    if denominator == 0:
        return 0.0
    
    return numerator / denominator


# Demonstrate the PPV problem
print("The PPV Problem: 95% Sensitivity + 95% Specificity")
print("=" * 55)
print("\n| Prevalence | PPV    | % False Alerts |")
print("|------------|--------|----------------|")

for prev in [0.50, 0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.001]:
    ppv = calculate_ppv(0.95, 0.95, prev)
    false_alert_rate = 1 - ppv
    print(f"| {prev*100:>6.1f}%    | {ppv*100:>5.1f}% | {false_alert_rate*100:>12.1f}%  |")

print("\n Key insight: At 1% prevalence, even excellent test performance")
print("    results in >80% false positive alerts!")

In [ ]:
# Interactive PPV visualization across different conditions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: PPV vs Prevalence for different specificity levels
ax = axes[0]
prevalences = np.linspace(0.001, 0.30, 100)
sensitivity = 0.90

for spec, color, style in [(0.99, COLORS['safe'], '-'), 
                            (0.95, COLORS['reference'], '--'),
                            (0.90, COLORS['alert'], '-.'),
                            (0.80, COLORS['current'], ':')]:
    ppvs = [calculate_ppv(sensitivity, spec, p) for p in prevalences]
    ax.plot(prevalences * 100, np.array(ppvs) * 100, style, 
            label=f'Specificity = {spec:.0%}', linewidth=2.5, color=color)

ax.axhline(y=50, color='gray', linestyle=':', alpha=0.5, linewidth=1)
ax.text(25, 52, 'Coin flip (50% PPV)', fontsize=9, color='gray')
ax.set_xlabel('Prevalence (%)', fontsize=12)
ax.set_ylabel('Positive Predictive Value (%)', fontsize=12)
ax.set_title(f'PPV vs Prevalence (Sensitivity = {sensitivity:.0%})', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0, 30)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

# Plot 2: PPV heatmap across sensitivity and specificity
ax = axes[1]
sens_range = np.linspace(0.70, 0.99, 30)
spec_range = np.linspace(0.70, 0.99, 30)
prevalence = 0.05  # 5% prevalence

ppv_matrix = np.zeros((len(spec_range), len(sens_range)))
for i, spec in enumerate(spec_range):
    for j, sens in enumerate(sens_range):
        ppv_matrix[i, j] = calculate_ppv(sens, spec, prevalence) * 100

im = ax.imshow(ppv_matrix, origin='lower', aspect='auto', cmap='RdYlGn',
               extent=[70, 99, 70, 99], vmin=0, vmax=100)
ax.set_xlabel('Sensitivity (%)', fontsize=12)
ax.set_ylabel('Specificity (%)', fontsize=12)
ax.set_title(f'PPV Heatmap (Prevalence = {prevalence:.0%})', fontsize=13, fontweight='bold')
cbar = plt.colorbar(im, ax=ax, label='PPV (%)')

# Add contour lines
contours = ax.contour(sens_range * 100, spec_range * 100, ppv_matrix, 
                       levels=[25, 50, 75], colors='black', linewidths=1)
ax.clabel(contours, inline=True, fontsize=9, fmt='%d%%')

# Plot 3: False positive rate per 1000 patients
ax = axes[2]
prevalences = [0.01, 0.05, 0.10, 0.20]
x_pos = np.arange(len(prevalences))
width = 0.25

sens = 0.90
for i, spec in enumerate([0.85, 0.95, 0.99]):
    fp_rates = []
    for prev in prevalences:
        # False positives per 1000 patients
        n_healthy = 1000 * (1 - prev)
        fp = n_healthy * (1 - spec)
        fp_rates.append(fp)
    
    color = [COLORS['current'], COLORS['alert'], COLORS['safe']][i]
    ax.bar(x_pos + (i-1)*width, fp_rates, width, 
           label=f'Specificity = {spec:.0%}', color=color, alpha=0.8)

ax.set_xlabel('Prevalence', fontsize=12)
ax.set_ylabel('False Positives per 1,000 Patients', fontsize=12)
ax.set_title('False Alert Burden (Sensitivity = 90%)', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{p:.0%}' for p in prevalences])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n PPV visualizations complete")

### Clinical Implications

The PPV problem has profound implications for alert design:

| Prevalence Setting | Example | PPV Challenge |
|-------------------|---------|---------------|
| High (>20%) | ICU sepsis screening | PPV acceptable; sensitivity prioritized |
| Moderate (5-20%) | ED chest pain workup | Balance needed; tiered alerts helpful |
| Low (1-5%) | Hospital-wide deterioration | False alarms dominate; high specificity essential |
| Very low (<1%) | Rare disease screening | PPV problematic; consider rule-out approaches |

**Design principle:** Before deploying any alert, calculate the expected PPV in your specific population. If PPV < 20%, reconsider the alert design.

---

## 2. Simulating a Sepsis Alert System

We will simulate a sepsis early warning system deployed in a hospital. This system generates alerts based on a continuous risk score, and we will explore how threshold choices affect alert burden and clinical utility.

In [ ]:
def generate_hospital_patients(n_patients, sepsis_prevalence=0.03, seed=None):
    """
    Generate synthetic patient data for sepsis alert simulation.
    
    Parameters:
    -----------
    n_patients : int
        Number of patients per day
    sepsis_prevalence : float
        True sepsis rate in population
    seed : int, optional
        Random seed for reproducibility
    
    Returns:
    --------
    pd.DataFrame with patient data, true sepsis status, and risk scores
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Generate true sepsis status
    sepsis = np.random.random(n_patients) < sepsis_prevalence
    
    # Generate risk scores (model output)
    # Sepsis patients have higher scores (imperfect discrimination)
    risk_scores = np.where(
        sepsis,
        np.random.beta(5, 2, n_patients),  # Sepsis: higher scores
        np.random.beta(2, 5, n_patients)   # Non-sepsis: lower scores
    )
    
    # Patient characteristics
    ages = np.random.normal(60, 18, n_patients).clip(18, 95)
    unit = np.random.choice(['ICU', 'Step-down', 'Med-Surg', 'ED'], 
                           n_patients, p=[0.15, 0.20, 0.45, 0.20])
    
    return pd.DataFrame({
        'patient_id': range(n_patients),
        'age': ages,
        'unit': unit,
        'sepsis': sepsis.astype(int),
        'risk_score': risk_scores
    })


def evaluate_alert_threshold(df, threshold):
    """
    Evaluate alert system performance at a given threshold.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Patient data with 'risk_score' and 'sepsis' columns
    threshold : float
        Alert threshold (0-1)
    
    Returns:
    --------
    dict : Performance metrics
    """
    alerts = df['risk_score'] >= threshold
    sepsis = df['sepsis'] == 1
    
    tp = (alerts & sepsis).sum()
    fp = (alerts & ~sepsis).sum()
    tn = (~alerts & ~sepsis).sum()
    fn = (~alerts & sepsis).sum()
    
    total_alerts = tp + fp
    total_patients = len(df)
    
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    
    return {
        'threshold': threshold,
        'total_alerts': total_alerts,
        'alerts_per_100': total_alerts / total_patients * 100,
        'true_positives': tp,
        'false_positives': fp,
        'false_negatives': fn,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'cases_caught': tp,
        'cases_missed': fn
    }


# Generate one day of hospital data
df_patients = generate_hospital_patients(n_patients=10000, sepsis_prevalence=0.03, seed=42)

print(f"Hospital Simulation: {len(df_patients):,} patients")
print(f"True sepsis cases: {df_patients['sepsis'].sum()} ({df_patients['sepsis'].mean():.1%} prevalence)")
print(f"\nRisk Score Distribution:")
print(df_patients.groupby('sepsis')['risk_score'].describe().round(3))

In [ ]:
# Evaluate multiple thresholds
thresholds = np.linspace(0.1, 0.9, 17)
threshold_results = [evaluate_alert_threshold(df_patients, t) for t in thresholds]
df_thresholds = pd.DataFrame(threshold_results)

# Display key thresholds
print("Alert System Performance at Different Thresholds")
print("=" * 90)
display_cols = ['threshold', 'total_alerts', 'sensitivity', 'specificity', 'ppv', 'cases_caught', 'cases_missed']
print(df_thresholds[display_cols].to_string(index=False, float_format=lambda x: f'{x:.2f}' if isinstance(x, float) else str(x)))

In [ ]:
# Comprehensive visualization of threshold tradeoffs
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Plot 1: Risk score distributions
ax = axes[0, 0]
df_sepsis = df_patients[df_patients['sepsis'] == 1]['risk_score']
df_no_sepsis = df_patients[df_patients['sepsis'] == 0]['risk_score']

ax.hist(df_no_sepsis, bins=50, alpha=0.6, density=True, 
        label=f'No Sepsis (n={len(df_no_sepsis):,})', color=COLORS['safe'])
ax.hist(df_sepsis, bins=50, alpha=0.6, density=True, 
        label=f'Sepsis (n={len(df_sepsis)})', color=COLORS['current'])
ax.axvline(x=0.4, color=COLORS['alert'], linestyle='--', linewidth=2, label='Example threshold (0.4)')
ax.set_xlabel('Risk Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Risk Score Distribution by Outcome', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Plot 2: Alerts per day vs threshold
ax = axes[0, 1]
ax.plot(df_thresholds['threshold'], df_thresholds['total_alerts'], 'o-', 
        color=COLORS['reference'], linewidth=2, markersize=8)
ax.axhline(y=100, color=COLORS['alert'], linestyle='--', linewidth=1.5, label='Manageable: 100 alerts')
ax.axhline(y=500, color=COLORS['current'], linestyle=':', linewidth=1.5, label='Burdensome: 500 alerts')
ax.set_xlabel('Alert Threshold', fontsize=12)
ax.set_ylabel('Total Alerts per Day', fontsize=12)
ax.set_title('Alert Volume vs Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Sensitivity vs Specificity tradeoff
ax = axes[1, 0]
ax.plot(df_thresholds['threshold'], df_thresholds['sensitivity'], 'o-', 
        color=COLORS['current'], linewidth=2, markersize=8, label='Sensitivity')
ax.plot(df_thresholds['threshold'], df_thresholds['specificity'], 's-', 
        color=COLORS['safe'], linewidth=2, markersize=8, label='Specificity')
ax.plot(df_thresholds['threshold'], df_thresholds['ppv'], '^-', 
        color=COLORS['alert'], linewidth=2, markersize=8, label='PPV')
ax.set_xlabel('Alert Threshold', fontsize=12)
ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Performance Metrics vs Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# Plot 4: Caught vs Missed cases
ax = axes[1, 1]
total_cases = df_patients['sepsis'].sum()
ax.fill_between(df_thresholds['threshold'], 0, df_thresholds['cases_caught'], 
                alpha=0.6, color=COLORS['safe'], label='Cases Caught')
ax.fill_between(df_thresholds['threshold'], df_thresholds['cases_caught'], total_cases, 
                alpha=0.6, color=COLORS['current'], label='Cases Missed')
ax.set_xlabel('Alert Threshold', fontsize=12)
ax.set_ylabel('Number of Sepsis Cases', fontsize=12)
ax.set_title('Detection Coverage vs Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, total_cases * 1.1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Threshold analysis visualizations complete")

---

## 3. Alert Fatigue Modeling

Alert fatigue is the phenomenon where clinicians become desensitized to alerts due to high volumes, leading to delayed or absent responses even to important alerts. This is not a character flaw---it is a predictable human response to information overload.

### The Fatigue Model

We model clinician response rate as an exponential decay function of alert volume:

$$\text{Response Rate} = \text{Base Rate} \times e^{-k \times \text{Alerts per Shift}}$$

Where:
- **Base Rate**: Response rate with minimal alerts (~95%)
- **k**: Fatigue decay constant (how quickly fatigue sets in)
- **Alerts per Shift**: Number of alerts a clinician sees during their shift

In [ ]:
def fatigue_response_rate(alerts_per_shift, base_rate=0.95, decay_constant=0.015):
    """
    Calculate clinician response rate accounting for alert fatigue.
    
    Parameters:
    -----------
    alerts_per_shift : int or array
        Number of alerts seen during shift
    base_rate : float
        Response rate with minimal alerts (default 0.95)
    decay_constant : float
        How quickly fatigue sets in (default 0.015)
    
    Returns:
    --------
    float or array : Response rate(s)
    """
    return base_rate * np.exp(-decay_constant * alerts_per_shift)


def simulate_shift_outcomes(n_patients_per_shift, sepsis_prevalence, threshold, 
                           n_clinicians=10, base_response=0.95, decay_k=0.015, seed=None):
    """
    Simulate outcomes for a shift including alert fatigue effects.
    
    Returns:
    --------
    dict : Shift outcome statistics
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Generate patients
    df = generate_hospital_patients(n_patients_per_shift, sepsis_prevalence, seed)
    
    # Generate alerts
    df['alert'] = df['risk_score'] >= threshold
    total_alerts = df['alert'].sum()
    alerts_per_clinician = total_alerts / n_clinicians
    
    # Calculate response rate with fatigue
    response_rate = fatigue_response_rate(alerts_per_clinician, base_response, decay_k)
    
    # Simulate responses to alerts
    df['responded'] = df['alert'] & (np.random.random(len(df)) < response_rate)
    
    # Calculate outcomes
    true_alerts = df['alert'] & (df['sepsis'] == 1)
    responded_true = df['responded'] & (df['sepsis'] == 1)
    
    return {
        'total_patients': n_patients_per_shift,
        'total_sepsis': df['sepsis'].sum(),
        'total_alerts': total_alerts,
        'alerts_per_clinician': alerts_per_clinician,
        'response_rate': response_rate,
        'true_alerts': true_alerts.sum(),
        'true_alerts_responded': responded_true.sum(),
        'true_alerts_missed_fatigue': true_alerts.sum() - responded_true.sum(),
        'detection_rate_pre_fatigue': true_alerts.sum() / df['sepsis'].sum() if df['sepsis'].sum() > 0 else 0,
        'detection_rate_post_fatigue': responded_true.sum() / df['sepsis'].sum() if df['sepsis'].sum() > 0 else 0
    }


# Visualize fatigue curve
alerts_range = np.linspace(0, 200, 100)
response_rates = fatigue_response_rate(alerts_range)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Fatigue curve
ax = axes[0]
ax.plot(alerts_range, response_rates * 100, color=COLORS['reference'], linewidth=3)
ax.axhline(y=50, color=COLORS['alert'], linestyle='--', linewidth=1.5, label='50% response rate')
ax.axvline(x=46, color='gray', linestyle=':', alpha=0.7)  # ~50% response point
ax.fill_between(alerts_range, 0, response_rates * 100, alpha=0.2, color=COLORS['reference'])
ax.set_xlabel('Alerts per Shift', fontsize=12)
ax.set_ylabel('Response Rate (%)', fontsize=12)
ax.set_title('Alert Fatigue: Response Rate Decay', fontsize=13, fontweight='bold')
ax.set_xlim(0, 200)
ax.set_ylim(0, 100)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate key points
ax.annotate('Fatigue begins', xy=(20, fatigue_response_rate(20)*100), 
            xytext=(40, 90), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='gray'))
ax.annotate('Critical fatigue zone', xy=(100, fatigue_response_rate(100)*100), 
            xytext=(120, 40), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='gray'))

# Plot 2: Different decay constants
ax = axes[1]
for k, label, color in [(0.005, 'Resilient (k=0.005)', COLORS['safe']),
                         (0.015, 'Typical (k=0.015)', COLORS['reference']),
                         (0.030, 'Sensitive (k=0.030)', COLORS['current'])]:
    rates = fatigue_response_rate(alerts_range, decay_constant=k)
    ax.plot(alerts_range, rates * 100, linewidth=2.5, label=label, color=color)

ax.set_xlabel('Alerts per Shift', fontsize=12)
ax.set_ylabel('Response Rate (%)', fontsize=12)
ax.set_title('Fatigue Sensitivity Varies by Context', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 200)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Fatigue model visualization complete")

In [ ]:
# Simulate the "cry wolf" effect over multiple shifts
def simulate_multiple_shifts(n_shifts, n_patients_per_shift, sepsis_prevalence, 
                             threshold, n_clinicians=10, seed=42):
    """
    Simulate multiple shifts to observe cumulative fatigue effects.
    """
    np.random.seed(seed)
    results = []
    
    for shift in range(n_shifts):
        outcome = simulate_shift_outcomes(
            n_patients_per_shift, sepsis_prevalence, threshold,
            n_clinicians=n_clinicians, seed=seed + shift
        )
        outcome['shift'] = shift + 1
        results.append(outcome)
    
    return pd.DataFrame(results)


# Compare low vs high alert burden scenarios
print("Simulating 30 shifts with different alert thresholds...")
print("\n")

# High alert burden (low threshold)
df_high_burden = simulate_multiple_shifts(
    n_shifts=30, n_patients_per_shift=500, sepsis_prevalence=0.03,
    threshold=0.25, n_clinicians=5, seed=42
)

# Low alert burden (high threshold)
df_low_burden = simulate_multiple_shifts(
    n_shifts=30, n_patients_per_shift=500, sepsis_prevalence=0.03,
    threshold=0.50, n_clinicians=5, seed=42
)

print("High Alert Burden (Threshold=0.25):")
print(f"  Average alerts per clinician per shift: {df_high_burden['alerts_per_clinician'].mean():.1f}")
print(f"  Average response rate: {df_high_burden['response_rate'].mean():.1%}")
print(f"  True cases caught (pre-fatigue): {df_high_burden['detection_rate_pre_fatigue'].mean():.1%}")
print(f"  True cases caught (post-fatigue): {df_high_burden['detection_rate_post_fatigue'].mean():.1%}")
print(f"  Cases missed due to fatigue: {df_high_burden['true_alerts_missed_fatigue'].sum()}")

print("\nLow Alert Burden (Threshold=0.50):")
print(f"  Average alerts per clinician per shift: {df_low_burden['alerts_per_clinician'].mean():.1f}")
print(f"  Average response rate: {df_low_burden['response_rate'].mean():.1%}")
print(f"  True cases caught (pre-fatigue): {df_low_burden['detection_rate_pre_fatigue'].mean():.1%}")
print(f"  True cases caught (post-fatigue): {df_low_burden['detection_rate_post_fatigue'].mean():.1%}")
print(f"  Cases missed due to threshold: {(df_low_burden['total_sepsis'] - df_low_burden['true_alerts']).sum()}")

In [ ]:
# Visualize the cry wolf paradox
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Alert burden comparison
ax = axes[0]
x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [df_high_burden['alerts_per_clinician'].mean(), 
                     df_low_burden['alerts_per_clinician'].mean()],
       width, label='Alerts per Clinician', color=COLORS['alert'])
ax.bar(x + width/2, [df_high_burden['response_rate'].mean() * 100, 
                     df_low_burden['response_rate'].mean() * 100],
       width, label='Response Rate (%)', color=COLORS['safe'])
ax.set_xticks(x)
ax.set_xticklabels(['High Burden\n(Threshold=0.25)', 'Low Burden\n(Threshold=0.50)'])
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Alert Burden vs Response Rate', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Detection rates
ax = axes[1]
metrics = ['Pre-Fatigue\nDetection', 'Post-Fatigue\nDetection']
high_vals = [df_high_burden['detection_rate_pre_fatigue'].mean() * 100,
             df_high_burden['detection_rate_post_fatigue'].mean() * 100]
low_vals = [df_low_burden['detection_rate_pre_fatigue'].mean() * 100,
            df_low_burden['detection_rate_post_fatigue'].mean() * 100]

x = np.arange(len(metrics))
ax.bar(x - width/2, high_vals, width, label='High Burden', color=COLORS['current'])
ax.bar(x + width/2, low_vals, width, label='Low Burden', color=COLORS['reference'])
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Detection Rate (%)', fontsize=12)
ax.set_title('The Cry Wolf Paradox', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

# Add annotation
ax.annotate('Fatigue erodes\ninitial advantage', 
            xy=(0.5, high_vals[1]), xytext=(0.8, high_vals[1] + 15),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='gray'))

# Plot 3: Cumulative missed cases over shifts
ax = axes[2]
high_cumulative = df_high_burden['true_alerts_missed_fatigue'].cumsum()
# For low burden, missed cases are those below threshold
low_missed_threshold = df_low_burden['total_sepsis'] - df_low_burden['true_alerts']
low_cumulative = low_missed_threshold.cumsum()

ax.plot(df_high_burden['shift'], high_cumulative, 'o-', 
        color=COLORS['current'], linewidth=2, markersize=6, label='High Burden (fatigue)')
ax.plot(df_low_burden['shift'], low_cumulative, 's-', 
        color=COLORS['reference'], linewidth=2, markersize=6, label='Low Burden (threshold)')
ax.set_xlabel('Shift Number', fontsize=12)
ax.set_ylabel('Cumulative Missed Cases', fontsize=12)
ax.set_title('Cumulative Missed Sepsis Cases', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n The cry wolf paradox: More sensitive alerts can miss MORE cases due to fatigue!")

### Key Insight: The Cry Wolf Paradox

The simulation reveals a counterintuitive finding: **more sensitive alert systems can actually catch fewer true cases** when fatigue is considered.

| Strategy | Theoretical Detection | Actual Detection (with Fatigue) |
|----------|----------------------|--------------------------------|
| High sensitivity, low threshold | High (catches most cases) | Degraded (fatigue causes misses) |
| Moderate sensitivity, higher threshold | Moderate (misses some cases) | Preserved (clinicians respond) |

**Implication:** Alert design must account for human factors, not just statistical performance.

---

## 4. Threshold Optimization

How do we find the optimal threshold that balances detection rate, alert burden, and fatigue effects? We need a utility function that captures clinical priorities.

In [ ]:
def calculate_clinical_utility(df_patients, threshold, n_clinicians=10,
                               cost_missed_case=100, cost_false_alarm=1,
                               cost_fatigue_factor=0.5):
    """
    Calculate clinical utility score for a given threshold.
    
    Parameters:
    -----------
    df_patients : pd.DataFrame
        Patient data
    threshold : float
        Alert threshold
    n_clinicians : int
        Number of clinicians per shift
    cost_missed_case : float
        Relative cost of missing a true case
    cost_false_alarm : float
        Relative cost of a false alarm
    cost_fatigue_factor : float
        Additional cost multiplier for fatigue-induced misses
    
    Returns:
    --------
    dict : Utility metrics
    """
    # Basic metrics
    alerts = df_patients['risk_score'] >= threshold
    sepsis = df_patients['sepsis'] == 1
    
    true_positives = (alerts & sepsis).sum()
    false_positives = (alerts & ~sepsis).sum()
    false_negatives = (~alerts & sepsis).sum()
    
    # Fatigue effects
    alerts_per_clinician = alerts.sum() / n_clinicians
    response_rate = fatigue_response_rate(alerts_per_clinician)
    
    # Effective catches accounting for fatigue
    effective_catches = true_positives * response_rate
    fatigue_misses = true_positives * (1 - response_rate)
    
    # Total misses = threshold misses + fatigue misses
    total_misses = false_negatives + fatigue_misses
    
    # Calculate utility (negative costs)
    cost_threshold_misses = false_negatives * cost_missed_case
    cost_fatigue_misses = fatigue_misses * cost_missed_case * (1 + cost_fatigue_factor)
    cost_false_alarms = false_positives * cost_false_alarm
    
    total_cost = cost_threshold_misses + cost_fatigue_misses + cost_false_alarms
    
    # Benefit from caught cases
    benefit_caught = effective_catches * cost_missed_case  # Avoided cost
    
    net_utility = benefit_caught - total_cost
    
    return {
        'threshold': threshold,
        'alerts': alerts.sum(),
        'alerts_per_clinician': alerts_per_clinician,
        'response_rate': response_rate,
        'true_positives': true_positives,
        'effective_catches': effective_catches,
        'threshold_misses': false_negatives,
        'fatigue_misses': fatigue_misses,
        'total_misses': total_misses,
        'false_alarms': false_positives,
        'total_cost': total_cost,
        'net_utility': net_utility
    }


# Grid search over thresholds
thresholds = np.linspace(0.15, 0.85, 71)
utility_results = [calculate_clinical_utility(df_patients, t) for t in thresholds]
df_utility = pd.DataFrame(utility_results)

# Find optimal threshold
optimal_idx = df_utility['net_utility'].idxmax()
optimal_threshold = df_utility.loc[optimal_idx, 'threshold']

print("Threshold Optimization Results")
print("=" * 60)
print(f"\nOptimal threshold: {optimal_threshold:.2f}")
print(f"\nAt optimal threshold:")
print(f"  Alerts per day: {df_utility.loc[optimal_idx, 'alerts']:.0f}")
print(f"  Alerts per clinician: {df_utility.loc[optimal_idx, 'alerts_per_clinician']:.1f}")
print(f"  Response rate: {df_utility.loc[optimal_idx, 'response_rate']:.1%}")
print(f"  Effective catches: {df_utility.loc[optimal_idx, 'effective_catches']:.1f}")
print(f"  Total misses: {df_utility.loc[optimal_idx, 'total_misses']:.1f}")

In [ ]:
# Visualization of optimization
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# Plot 1: Net utility curve
ax = axes[0, 0]
ax.plot(df_utility['threshold'], df_utility['net_utility'], 
        color=COLORS['reference'], linewidth=3)
ax.axvline(x=optimal_threshold, color=COLORS['safe'], linestyle='--', 
           linewidth=2, label=f'Optimal: {optimal_threshold:.2f}')
ax.scatter([optimal_threshold], [df_utility.loc[optimal_idx, 'net_utility']], 
           color=COLORS['safe'], s=150, zorder=5, edgecolor='black')
ax.set_xlabel('Alert Threshold', fontsize=12)
ax.set_ylabel('Net Clinical Utility', fontsize=12)
ax.set_title('Utility Optimization', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 2: Breakdown of misses
ax = axes[0, 1]
ax.fill_between(df_utility['threshold'], 0, df_utility['threshold_misses'], 
                alpha=0.6, color=COLORS['current'], label='Threshold Misses')
ax.fill_between(df_utility['threshold'], df_utility['threshold_misses'], 
                df_utility['total_misses'], 
                alpha=0.6, color=COLORS['alert'], label='Fatigue Misses')
ax.axvline(x=optimal_threshold, color=COLORS['safe'], linestyle='--', linewidth=2)
ax.set_xlabel('Alert Threshold', fontsize=12)
ax.set_ylabel('Missed Cases', fontsize=12)
ax.set_title('Sources of Missed Cases', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Effective detection rate
ax = axes[1, 0]
total_sepsis = df_patients['sepsis'].sum()
theoretical_rate = df_utility['true_positives'] / total_sepsis * 100
effective_rate = df_utility['effective_catches'] / total_sepsis * 100

ax.plot(df_utility['threshold'], theoretical_rate, '--', 
        color=COLORS['reference'], linewidth=2, label='Theoretical (no fatigue)')
ax.plot(df_utility['threshold'], effective_rate, '-', 
        color=COLORS['current'], linewidth=3, label='Effective (with fatigue)')
ax.fill_between(df_utility['threshold'], effective_rate, theoretical_rate, 
                alpha=0.3, color=COLORS['alert'], label='Fatigue loss')
ax.axvline(x=optimal_threshold, color=COLORS['safe'], linestyle='--', linewidth=2)
ax.set_xlabel('Alert Threshold', fontsize=12)
ax.set_ylabel('Detection Rate (%)', fontsize=12)
ax.set_title('Theoretical vs Effective Detection', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

# Plot 4: Tradeoff curve (alerts vs catches)
ax = axes[1, 1]
ax.plot(df_utility['alerts'], df_utility['effective_catches'], 'o-', 
        color=COLORS['reference'], linewidth=2, markersize=4, alpha=0.7)
opt_alerts = df_utility.loc[optimal_idx, 'alerts']
opt_catches = df_utility.loc[optimal_idx, 'effective_catches']
ax.scatter([opt_alerts], [opt_catches], color=COLORS['safe'], s=200, 
           zorder=5, edgecolor='black', label=f'Optimal ({opt_alerts:.0f} alerts)')
ax.set_xlabel('Total Alerts per Day', fontsize=12)
ax.set_ylabel('Effective Cases Caught', fontsize=12)
ax.set_title('Alert Burden vs Detection Tradeoff', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Mark diminishing returns region
ax.axvspan(opt_alerts, df_utility['alerts'].max(), alpha=0.1, color='red')
ax.text(opt_alerts + 200, opt_catches - 20, 'Diminishing\nreturns', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

print("\n Threshold optimization complete")

---

## 5. Context-Specific Alert Design

Different clinical contexts have different prevalences, consequences, and alert tolerances. A one-size-fits-all threshold is inappropriate.

In [ ]:
# Define clinical contexts
clinical_contexts = {
    'ICU': {
        'prevalence': 0.12,
        'patients_per_shift': 200,
        'clinicians': 15,
        'cost_missed': 150,  # High consequence
        'cost_false_alarm': 0.5,  # Staff used to alerts
        'description': 'High acuity, high prevalence, alert-tolerant staff'
    },
    'Step-down': {
        'prevalence': 0.05,
        'patients_per_shift': 400,
        'clinicians': 10,
        'cost_missed': 120,
        'cost_false_alarm': 1,
        'description': 'Moderate acuity, moderate prevalence'
    },
    'Med-Surg': {
        'prevalence': 0.02,
        'patients_per_shift': 800,
        'clinicians': 8,
        'cost_missed': 100,
        'cost_false_alarm': 2,  # Staff less alert-tolerant
        'description': 'Lower acuity, low prevalence, less alert tolerance'
    },
    'Outpatient': {
        'prevalence': 0.005,
        'patients_per_shift': 500,
        'clinicians': 5,
        'cost_missed': 80,
        'cost_false_alarm': 5,  # Very low alert tolerance
        'description': 'Low acuity, very low prevalence, minimal alert tolerance'
    }
}


def optimize_for_context(context_name, context_params, n_patients=5000, seed=42):
    """
    Find optimal threshold for a specific clinical context.
    """
    # Generate context-specific patient population
    df = generate_hospital_patients(
        n_patients, 
        sepsis_prevalence=context_params['prevalence'],
        seed=seed
    )
    
    # Search for optimal threshold
    thresholds = np.linspace(0.15, 0.85, 71)
    best_utility = -np.inf
    best_threshold = 0.5
    best_metrics = None
    
    for t in thresholds:
        metrics = calculate_clinical_utility(
            df, t,
            n_clinicians=context_params['clinicians'],
            cost_missed_case=context_params['cost_missed'],
            cost_false_alarm=context_params['cost_false_alarm']
        )
        
        if metrics['net_utility'] > best_utility:
            best_utility = metrics['net_utility']
            best_threshold = t
            best_metrics = metrics
    
    return {
        'context': context_name,
        'optimal_threshold': best_threshold,
        'prevalence': context_params['prevalence'],
        'alerts_per_clinician': best_metrics['alerts_per_clinician'],
        'response_rate': best_metrics['response_rate'],
        'effective_detection': best_metrics['effective_catches'] / df['sepsis'].sum(),
        'ppv': best_metrics['true_positives'] / best_metrics['alerts'] if best_metrics['alerts'] > 0 else 0
    }


# Optimize for each context
print("Optimizing thresholds for each clinical context...")
context_results = []
for name, params in clinical_contexts.items():
    result = optimize_for_context(name, params)
    context_results.append(result)

df_contexts = pd.DataFrame(context_results)

print("\nContext-Specific Optimal Thresholds")
print("=" * 90)
print(df_contexts.to_string(index=False, float_format=lambda x: f'{x:.2f}' if isinstance(x, float) else str(x)))

In [ ]:
# Visualize context-specific recommendations
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

contexts = df_contexts['context'].tolist()
x_pos = np.arange(len(contexts))

# Plot 1: Optimal thresholds by context
ax = axes[0]
colors = [COLORS['current'], COLORS['alert'], COLORS['reference'], COLORS['safe']]
bars = ax.bar(x_pos, df_contexts['optimal_threshold'], color=colors, alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(contexts, rotation=15, ha='right')
ax.set_ylabel('Optimal Threshold', fontsize=12)
ax.set_title('Optimal Threshold by Context', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, df_contexts['optimal_threshold']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

# Plot 2: PPV comparison
ax = axes[1]
bars = ax.bar(x_pos, df_contexts['ppv'] * 100, color=colors, alpha=0.8)
ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, label='Coin flip')
ax.set_xticks(x_pos)
ax.set_xticklabels(contexts, rotation=15, ha='right')
ax.set_ylabel('PPV (%)', fontsize=12)
ax.set_title('Positive Predictive Value by Context', fontsize=13, fontweight='bold')
ax.set_ylim(0, 100)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Detection rate vs prevalence
ax = axes[2]
ax.scatter(df_contexts['prevalence'] * 100, df_contexts['effective_detection'] * 100,
           s=200, c=colors, edgecolors='black', linewidth=2, zorder=5)

for i, ctx in enumerate(contexts):
    ax.annotate(ctx, 
                xy=(df_contexts.loc[i, 'prevalence'] * 100, 
                    df_contexts.loc[i, 'effective_detection'] * 100),
                xytext=(5, 5), textcoords='offset points', fontsize=10)

ax.set_xlabel('Prevalence (%)', fontsize=12)
ax.set_ylabel('Effective Detection Rate (%)', fontsize=12)
ax.set_title('Detection vs Prevalence by Context', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n Context-specific analysis complete")

### Context-Specific Decision Framework

| Clinical Context | Prevalence | Recommended Threshold | Alert Strategy | Key Consideration |
|-----------------|------------|----------------------|----------------|-------------------|
| **ICU** | High (>10%) | Lower (0.3-0.4) | Aggressive | Maximize sensitivity; staff can handle volume |
| **Step-down** | Moderate (5-10%) | Moderate (0.4-0.5) | Balanced | Balance sensitivity and specificity |
| **Med-Surg** | Low (2-5%) | Higher (0.5-0.6) | Conservative | Prioritize PPV; reduce fatigue |
| **Outpatient** | Very Low (<2%) | Highest (0.6-0.7) | Selective | Very high specificity required |

---

## 6. Alert Presentation Design

Beyond threshold selection, how alerts are presented significantly affects their clinical impact. This section explores tiered alerts, information content, and timing considerations.

In [ ]:
def create_tiered_alert_system(df_patients, thresholds_dict):
    """
    Create a tiered alert system with multiple risk levels.
    
    Parameters:
    -----------
    df_patients : pd.DataFrame
        Patient data with risk_score
    thresholds_dict : dict
        Dictionary with 'low', 'medium', 'high' threshold values
    
    Returns:
    --------
    pd.DataFrame with tier assignments
    """
    df = df_patients.copy()
    
    conditions = [
        df['risk_score'] >= thresholds_dict['high'],
        df['risk_score'] >= thresholds_dict['medium'],
        df['risk_score'] >= thresholds_dict['low']
    ]
    choices = ['HIGH', 'MEDIUM', 'LOW']
    
    df['alert_tier'] = np.select(conditions, choices, default='NONE')
    
    return df


# Define tiered thresholds
tier_thresholds = {
    'low': 0.30,    # Monitor
    'medium': 0.50, # Assess
    'high': 0.70    # Immediate action
}

# Apply tiered system
df_tiered = create_tiered_alert_system(df_patients, tier_thresholds)

# Analyze tiers
tier_analysis = df_tiered.groupby('alert_tier').agg({
    'patient_id': 'count',
    'sepsis': ['sum', 'mean'],
    'risk_score': 'mean'
}).round(3)

tier_analysis.columns = ['Count', 'Sepsis Cases', 'Sepsis Rate', 'Mean Risk']
tier_analysis = tier_analysis.reindex(['HIGH', 'MEDIUM', 'LOW', 'NONE'])

print("Tiered Alert System Analysis")
print("=" * 60)
print(f"\nThresholds: Low >= {tier_thresholds['low']}, Medium >= {tier_thresholds['medium']}, High >= {tier_thresholds['high']}")
print("\n")
print(tier_analysis.to_string())

In [ ]:
# Visualize tiered system
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

tier_colors = {'HIGH': COLORS['current'], 'MEDIUM': COLORS['alert'], 
               'LOW': COLORS['reference'], 'NONE': COLORS['safe']}

# Plot 1: Distribution of tiers
ax = axes[0]
tier_counts = df_tiered['alert_tier'].value_counts().reindex(['HIGH', 'MEDIUM', 'LOW', 'NONE'])
colors = [tier_colors[t] for t in tier_counts.index]
bars = ax.bar(tier_counts.index, tier_counts.values, color=colors, alpha=0.8)
ax.set_xlabel('Alert Tier', fontsize=12)
ax.set_ylabel('Number of Patients', fontsize=12)
ax.set_title('Patient Distribution by Alert Tier', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add counts
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
            f'{bar.get_height():.0f}', ha='center', fontsize=11)

# Plot 2: Sepsis rate by tier
ax = axes[1]
sepsis_by_tier = df_tiered.groupby('alert_tier')['sepsis'].mean().reindex(['HIGH', 'MEDIUM', 'LOW', 'NONE'])
colors = [tier_colors[t] for t in sepsis_by_tier.index]
bars = ax.bar(sepsis_by_tier.index, sepsis_by_tier.values * 100, color=colors, alpha=0.8)
ax.axhline(y=df_patients['sepsis'].mean() * 100, color='black', linestyle='--', 
           label='Overall prevalence')
ax.set_xlabel('Alert Tier', fontsize=12)
ax.set_ylabel('Sepsis Rate (%)', fontsize=12)
ax.set_title('Sepsis Rate by Alert Tier (PPV)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add percentages
for bar, val in zip(bars, sepsis_by_tier.values * 100):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Plot 3: Risk score distribution with tiers
ax = axes[2]
for tier in ['HIGH', 'MEDIUM', 'LOW', 'NONE']:
    subset = df_tiered[df_tiered['alert_tier'] == tier]['risk_score']
    if len(subset) > 0:
        ax.hist(subset, bins=30, alpha=0.5, label=tier, color=tier_colors[tier], density=True)

# Add threshold lines
for label, thresh in tier_thresholds.items():
    ax.axvline(x=thresh, color='black', linestyle=':', linewidth=1.5)
    ax.text(thresh + 0.01, ax.get_ylim()[1] * 0.9, label.upper(), fontsize=9, rotation=90)

ax.set_xlabel('Risk Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Risk Score Distribution by Tier', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("\n Tiered alert visualization complete")

### Alert Information Content

Effective alerts provide actionable information, not just risk scores. Each tier should have appropriate content:

In [ ]:
# Define alert templates
alert_templates = {
    'HIGH': {
        'header': 'SEPSIS ALERT - IMMEDIATE ASSESSMENT REQUIRED',
        'color': 'Red',
        'timing': 'Immediate (interruptive)',
        'action': 'Bedside assessment within 15 minutes',
        'content': [
            'Risk score and contributing factors',
            'Recent vital sign trends',
            'Recommended sepsis workup (lactate, cultures, CBC)',
            'One-click order set activation',
            'Escalation contacts'
        ]
    },
    'MEDIUM': {
        'header': 'SEPSIS WATCH - ENHANCED MONITORING',
        'color': 'Orange',
        'timing': 'Within 30 minutes (semi-interruptive)',
        'action': 'Chart review and assessment plan',
        'content': [
            'Risk score with trend',
            'Key abnormal values highlighted',
            'Suggested monitoring frequency',
            'Snooze/acknowledge options'
        ]
    },
    'LOW': {
        'header': 'SEPSIS RISK - MONITOR',
        'color': 'Yellow',
        'timing': 'During routine rounding (non-interruptive)',
        'action': 'Include in handoff',
        'content': [
            'Risk score displayed on dashboard',
            'Auto-escalation if score increases',
            'No immediate action required'
        ]
    }
}

print("Alert Tier Specifications")
print("=" * 70)
for tier, specs in alert_templates.items():
    print(f"\n[{tier}] {specs['header']}")
    print(f"   Color: {specs['color']}")
    print(f"   Timing: {specs['timing']}")
    print(f"   Required Action: {specs['action']}")
    print(f"   Alert Content:")
    for item in specs['content']:
        print(f"     - {item}")

### Timing Considerations: Batching vs. Immediate

Not every alert needs to interrupt workflow immediately. Strategic batching reduces cognitive load while preserving responsiveness for true emergencies.

| Alert Type | Timing Strategy | Rationale |
|-----------|-----------------|----------|
| High-risk alerts | Immediate, interruptive | Patient safety requires immediate response |
| Medium-risk alerts | Batched (every 30 min) or at transitions | Reduces interruptions while maintaining vigilance |
| Low-risk alerts | Passive dashboard, handoff integration | Awareness without workflow disruption |
| Informational | End-of-shift summary | Context for ongoing care planning |

---

## 7. Summary and Best Practices

### Key Takeaways

In [ ]:
# Create summary table
summary_data = {
    'Principle': [
        'PPV depends on prevalence',
        'Alert fatigue is predictable',
        'Context determines thresholds',
        'Tiered alerts reduce noise',
        'Utility optimization > sensitivity maximization',
        'Information content matters'
    ],
    'Key Insight': [
        'Even 95% sens/spec yields poor PPV at low prevalence',
        'Response rate decays exponentially with alert volume',
        'ICU tolerates more alerts than outpatient settings',
        'HIGH/MEDIUM/LOW tiers focus attention appropriately',
        'More sensitive is not always better when fatigue considered',
        'Actionable content increases response likelihood'
    ],
    'Design Recommendation': [
        'Calculate expected PPV before deployment',
        'Target <50 alerts per clinician per shift',
        'Optimize thresholds for each deployment context',
        'Reserve interruptive alerts for high-risk only',
        'Include fatigue in utility calculations',
        'Provide contributing factors and suggested actions'
    ]
}

df_summary = pd.DataFrame(summary_data)

print("Summary: Clinical Alert Design Best Practices")
print("=" * 100)
print("\n")

# Display as formatted table
for i, row in df_summary.iterrows():
    print(f"{i+1}. {row['Principle']}")
    print(f"   Insight: {row['Key Insight']}")
    print(f"   Recommendation: {row['Design Recommendation']}")
    print()

### Recommended Threshold Selection Process

1. **Characterize the deployment context**
   - Determine disease prevalence in your population
   - Assess staffing levels and alert tolerance
   - Understand consequences of missed cases vs. false alarms

2. **Calculate expected PPV**
   - Use Bayes' theorem with your model's sensitivity/specificity
   - If PPV < 20%, reconsider whether an alert is appropriate

3. **Model alert burden**
   - Estimate daily alert volume at candidate thresholds
   - Calculate alerts per clinician per shift
   - Apply fatigue model to estimate effective detection

4. **Optimize for clinical utility**
   - Define costs for missed cases, false alarms, and fatigue
   - Find threshold that maximizes net utility
   - Validate with clinician stakeholders

5. **Implement tiered alerting**
   - Use multiple thresholds for different urgency levels
   - Match alert presentation to tier
   - Provide actionable content

6. **Monitor and iterate**
   - Track response rates by tier
   - Monitor alert-to-intervention ratios
   - Adjust thresholds based on real-world performance

### Warning: The Danger of "Turning Up Sensitivity"

A common request from clinical leadership: "We missed a case. Turn up the sensitivity."

**Why this is dangerous:**

In [ ]:
# Demonstrate the cascade effect of increasing sensitivity
def project_sensitivity_increase(current_threshold, new_threshold, df_patients, n_clinicians=10):
    """
    Project the impact of lowering threshold to increase sensitivity.
    """
    current = calculate_clinical_utility(df_patients, current_threshold, n_clinicians)
    new = calculate_clinical_utility(df_patients, new_threshold, n_clinicians)
    
    return {
        'Metric': ['Threshold', 'Sensitivity', 'Daily Alerts', 'Alerts/Clinician', 
                   'Response Rate', 'Cases Caught (effective)', 'False Alarms', 'PPV'],
        'Before': [
            f'{current_threshold:.2f}',
            f'{current["true_positives"] / df_patients["sepsis"].sum():.1%}',
            f'{current["alerts"]:.0f}',
            f'{current["alerts_per_clinician"]:.1f}',
            f'{current["response_rate"]:.1%}',
            f'{current["effective_catches"]:.1f}',
            f'{current["false_alarms"]:.0f}',
            f'{current["true_positives"]/current["alerts"]:.1%}' if current['alerts'] > 0 else 'N/A'
        ],
        'After': [
            f'{new_threshold:.2f}',
            f'{new["true_positives"] / df_patients["sepsis"].sum():.1%}',
            f'{new["alerts"]:.0f}',
            f'{new["alerts_per_clinician"]:.1f}',
            f'{new["response_rate"]:.1%}',
            f'{new["effective_catches"]:.1f}',
            f'{new["false_alarms"]:.0f}',
            f'{new["true_positives"]/new["alerts"]:.1%}' if new['alerts'] > 0 else 'N/A'
        ],
        'Change': [
            f'{new_threshold - current_threshold:+.2f}',
            f'+{(new["true_positives"] - current["true_positives"]) / df_patients["sepsis"].sum():.1%}',
            f'+{new["alerts"] - current["alerts"]:.0f}',
            f'+{new["alerts_per_clinician"] - current["alerts_per_clinician"]:.1f}',
            f'{(new["response_rate"] - current["response_rate"]):.1%}',
            f'{new["effective_catches"] - current["effective_catches"]:+.1f}',
            f'+{new["false_alarms"] - current["false_alarms"]:.0f}',
            'Decreased'
        ]
    }

# Scenario: Lowering threshold from 0.5 to 0.3
comparison = project_sensitivity_increase(0.50, 0.30, df_patients)
df_comparison = pd.DataFrame(comparison)

print("Impact of 'Turning Up Sensitivity' (Threshold 0.50 -> 0.30)")
print("=" * 70)
print("\n")
print(df_comparison.to_string(index=False))
print("\n")
print("CAUTION: Despite higher theoretical sensitivity, effective catches may")
print("         DECREASE due to fatigue-induced response degradation!")

In [ ]:
# Final summary visualization
print("\n" + "="*70)
print("NOTEBOOK COMPLETE: Clinical Alert Design and Optimization")
print("="*70)
print("""
 Calculated PPV across sensitivity/specificity/prevalence combinations
 Simulated sepsis alert system with configurable thresholds
 Modeled alert fatigue and the cry wolf effect
 Optimized thresholds using clinical utility functions
 Designed context-specific alert strategies (ICU vs. outpatient)
 Created tiered alert framework (HIGH/MEDIUM/LOW)
 Defined alert content and timing recommendations

Key message: Alert design is not just about model performance.
Consider prevalence, fatigue, context, and clinical workflow.
More sensitive alerts can paradoxically catch FEWER cases.
""")

---

## 8. Exercises

1. **PPV Calculator**: Modify the `calculate_ppv` function to also return the number needed to screen (NNS) to find one true positive. How does NNS change with prevalence?

2. **Custom Fatigue Model**: The exponential decay model is simplified. Research alternative fatigue models (e.g., cumulative fatigue, recovery effects) and implement one. How does it change the optimal threshold?

3. **Multi-condition Alerts**: Extend the simulation to include alerts for multiple conditions (sepsis, AKI, respiratory failure). How should thresholds be coordinated to manage total alert burden?

4. **Subgroup Optimization**: Different patient populations may have different prevalences. Implement a system that uses patient characteristics (age, unit, comorbidities) to select threshold dynamically.

5. **Response Rate Tracking**: Design a monitoring system that tracks actual clinician response rates over time. What metrics would you use to detect emerging alert fatigue?

6. **Cost-Effectiveness Analysis**: Extend the utility function to include actual healthcare costs (delayed treatment, unnecessary workups, nursing time). What threshold is most cost-effective for your institution?

---

*This notebook accompanies Chapter 5 of AI in Healthcare Volume 2. For the full clinical and regulatory context around deployment and integration, please refer to the textbook.*